# Lab 3 - HiNetLite NPU v6c

- Model: `hinet_lite_npu_v6c_stable_skip_adapter_distill`
- Goal: improve v6b PSNR by stabilizing training while preserving the v6b NPU-friendly architecture.
- Expected input/output: `256x256x3` RGB.
- Student initialization: v6 best checkpoint only.
- Teacher: hinet_colab_v5 checkpoint for weak residual distillation only.
- Submission path: self-contained notebook; no repo-local imports.


## 1. Setup

Run the Drive mount cell first. It reuses an existing `/content/drive/MyDrive` mount when available; if `/content/drive` is occupied but not usable, it mounts Drive at a clean fallback path and the rest of the notebook follows that selected mount point. The notebook writes a new v6c run under the Lab 3 Drive `runs/` folder and keeps the existing v6/v6b/v5 artifacts untouched.


In [1]:
try:
    import onnx  # noqa: F401
    import onnxruntime  # noqa: F401
except ImportError:
    %pip install -q onnx onnxruntime


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 133.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 141.7 MB/s eta 0:00:0000:0100:01


In [2]:
from pathlib import Path

LAB3_DRIVE_REL = Path('Data 255 Class Spring 2026/Data 255/Lab 3')


def choose_or_mount_drive() -> Path:
    mount_candidates = [Path('/content/drive'), Path('/content/gdrive'), Path('/content/lab3_drive_mount')]
    for mount_point in mount_candidates:
        if (mount_point / 'MyDrive').exists():
            return mount_point
    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError('This notebook expects Google Colab for Drive mounting.') from exc
    first_mount = mount_candidates[0]
    try:
        drive.mount(str(first_mount), force_remount=False)
        return first_mount
    except ValueError as exc:
        if 'Mountpoint must not already contain files' not in str(exc):
            raise
        print({'drive_mount_warning': str(exc), 'mountpoint': str(first_mount), 'children': sorted(path.name for path in first_mount.iterdir()) if first_mount.exists() else []})
    for mount_point in mount_candidates[1:]:
        mount_point.mkdir(parents=True, exist_ok=True)
        if any(mount_point.iterdir()) and not (mount_point / 'MyDrive').exists():
            continue
        drive.mount(str(mount_point), force_remount=False)
        return mount_point
    raise RuntimeError('Unable to find an empty mount point for Google Drive. Restart the Colab runtime and run this cell first.')


DRIVE_MOUNT_POINT = choose_or_mount_drive()
DRIVE_ROOT_CANDIDATES = [
    DRIVE_MOUNT_POINT / 'MyDrive' / LAB3_DRIVE_REL,
    Path('/content/drive/MyDrive') / LAB3_DRIVE_REL,
    Path('/content/gdrive/MyDrive') / LAB3_DRIVE_REL,
    Path('/content/lab3_drive_mount/MyDrive') / LAB3_DRIVE_REL,
]
DRIVE_ROOT = next((path for path in DRIVE_ROOT_CANDIDATES if path.exists()), DRIVE_ROOT_CANDIDATES[0])
DRIVE_DATA_ROOT = DRIVE_ROOT / 'Data'
required_l3_dirs = [
    DRIVE_DATA_ROOT / 'LR_HR' / 'L3_HR_train1',
    DRIVE_DATA_ROOT / 'L3_LR' / 'L3_L3_train1',
    DRIVE_DATA_ROOT / 'L3_HR_valid',
    DRIVE_DATA_ROOT / 'L3_LR_valid',
]
mount_diagnostic = {
    'drive_mount_point': str(DRIVE_MOUNT_POINT),
    'drive_root': str(DRIVE_ROOT),
    'drive_root_exists': DRIVE_ROOT.exists(),
    'data_root': str(DRIVE_DATA_ROOT),
    'data_root_exists': DRIVE_DATA_ROOT.exists(),
    'data_root_children': sorted(path.name for path in DRIVE_DATA_ROOT.iterdir()) if DRIVE_DATA_ROOT.exists() else [],
    'required_l3_dirs': {str(path.relative_to(DRIVE_DATA_ROOT)) if DRIVE_DATA_ROOT.exists() else str(path): path.exists() for path in required_l3_dirs},
}
print(mount_diagnostic)
missing_l3_dirs = [path for path in required_l3_dirs if not path.exists()]
if missing_l3_dirs:
    raise FileNotFoundError({'missing_l3_dirs': [str(path) for path in missing_l3_dirs], 'mount_diagnostic': mount_diagnostic})


Mounted at /content/drive
{'drive_mount_point': '/content/drive', 'drive_root': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3', 'drive_root_exists': True, 'data_root': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/Data', 'data_root_exists': True, 'data_root_children': ['.DS_Store', 'HR_train', 'HR_val', 'L3_HR_valid', 'L3_LR', 'L3_LR_valid', 'LR_HR', 'LR_train', 'LR_val', 'course_files_export'], 'required_l3_dirs': {'LR_HR/L3_HR_train1': True, 'L3_LR/L3_L3_train1': True, 'L3_HR_valid': True, 'L3_LR_valid': True}}


In [3]:
from __future__ import annotations

import csv
import json
import math
import random
import shlex
import shutil
import subprocess
import sys
import time
from collections import Counter
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset

try:
    import onnx
except ImportError:
    onnx = None

try:
    import onnxruntime as ort
except ImportError:
    ort = None

try:
    BICUBIC_RESAMPLE = Image.Resampling.BICUBIC
except AttributeError:
    BICUBIC_RESAMPLE = Image.BICUBIC

try:
    FLIP_LEFT_RIGHT = Image.Transpose.FLIP_LEFT_RIGHT
    FLIP_TOP_BOTTOM = Image.Transpose.FLIP_TOP_BOTTOM
    ROTATE_90 = Image.Transpose.ROTATE_90
    ROTATE_180 = Image.Transpose.ROTATE_180
    ROTATE_270 = Image.Transpose.ROTATE_270
except AttributeError:
    FLIP_LEFT_RIGHT = Image.FLIP_LEFT_RIGHT
    FLIP_TOP_BOTTOM = Image.FLIP_TOP_BOTTOM
    ROTATE_90 = Image.ROTATE_90
    ROTATE_180 = Image.ROTATE_180
    ROTATE_270 = Image.ROTATE_270


def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def now_stamp() -> str:
    return datetime.now().strftime('%H%M%S_%d%m')


def make_unique_run_root(runs_root: Path, run_slug: str) -> Path:
    safe_slug = ''.join(ch if ch.isalnum() or ch in '_-' else '_' for ch in run_slug).strip('_')
    candidate = runs_root / f'{now_stamp()}_{safe_slug}'
    while candidate.exists():
        time.sleep(1.1)
        candidate = runs_root / f'{now_stamp()}_{safe_slug}'
    return candidate


DRIVE_MOUNT_POINT = Path(globals().get('DRIVE_MOUNT_POINT', '/content/drive'))
DRIVE_ROOT = Path(globals().get('DRIVE_ROOT', DRIVE_MOUNT_POINT / 'MyDrive' / 'Data 255 Class Spring 2026' / 'Data 255' / 'Lab 3'))
DRIVE_DATA_ROOT = DRIVE_ROOT / 'Data'
DRIVE_RUNS_ROOT = DRIVE_ROOT / 'runs'
IMAGE_SUFFIXES = {'.png', '.jpg', '.jpeg', '.bmp'}
EXPECTED_TRAIN_PAIRS = 2217
EXPECTED_VAL_PAIRS = 110
V6_STUDENT_BEST_REL = '192606_2404_v5_clean_no_block_residual/checkpoints/best.pth'
V5_TEACHER_BEST_REL = '162928_2404_v5_c44_b5_mixed_psnr/checkpoints/best.pth'
V6B_REFERENCE_BEST_VAL_PSNR = 24.02910700711337
V6_REFERENCE_BEST_VAL_PSNR = 23.930102400346236
V5_REFERENCE_BEST_VAL_PSNR = 24.13589654650007
V4_REFERENCE_BEST_VAL_PSNR = 24.326435361589706


@dataclass
class RunConfig:
    model_id: str = 'hinet_lite_npu_v6c_stable_skip_adapter_distill'
    run_slug: str = 'v6c_stable_skip_adapter_distill'
    seed: int = 1337
    eval_size: int = 256
    crop_size: int = 256
    batch_size: int = 8
    epochs: int = 120
    lr: float = 1e-4
    min_lr: float = 5e-6
    weight_decay: float = 5e-5
    warmup_epochs: int = 5
    grad_clip_norm: float = 0.5
    mse_weight: float = 0.9
    l1_weight: float = 0.1
    distill_weight: float = 0.05
    adapter_freeze_epochs: int = 3
    early_stop_patience: int = 20
    num_workers: int = 2
    use_amp: bool = False
    use_ema: bool = True
    ema_decay: float = 0.999
    selected_weight_source: str = 'ema'
    student_checkpoint_rel: str = V6_STUDENT_BEST_REL
    teacher_checkpoint_rel: str = V5_TEACHER_BEST_REL
    allow_missing_teacher: bool = True
    train_pair_limit: int | None = None
    val_pair_limit: int | None = None
    calibration_count: int = 32
    run_mxq_compile: bool = False
    mxq_command_template: str = ''


def resolve_run_path(relative_run_path: str | None) -> Path | None:
    if not relative_run_path:
        return None
    candidate = Path(relative_run_path)
    if candidate.is_absolute():
        return candidate
    return DRIVE_RUNS_ROOT / candidate


cfg = RunConfig()
if cfg.eval_size != 256 or cfg.crop_size != 256:
    raise ValueError('Lab 3 v6c requires 256x256 train/eval tensors.')
if abs((cfg.mse_weight + cfg.l1_weight) - 1.0) > 1e-8:
    raise ValueError('mse_weight + l1_weight must equal 1.0 for this recipe.')
if cfg.selected_weight_source not in {'ema', 'model'}:
    raise ValueError(f'Unsupported selected_weight_source: {cfg.selected_weight_source}')

set_global_seed(cfg.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
student_checkpoint_path = resolve_run_path(cfg.student_checkpoint_rel)
teacher_checkpoint_path = resolve_run_path(cfg.teacher_checkpoint_rel)
run_root = make_unique_run_root(DRIVE_RUNS_ROOT, cfg.run_slug)
checkpoints_dir = run_root / 'checkpoints'
exports_dir = run_root / 'exports'
calibration_dir = exports_dir / 'calibration'
for path in [run_root, checkpoints_dir, exports_dir, calibration_dir]:
    path.mkdir(parents=True, exist_ok=True)

print({
    'device': str(device),
    'run_root': str(run_root),
    'data_root': str(DRIVE_DATA_ROOT),
    'student_checkpoint_path': str(student_checkpoint_path),
    'teacher_checkpoint_path': str(teacher_checkpoint_path),
    'config': asdict(cfg),
})


{'device': 'cuda', 'run_root': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/011149_2504_v6c_stable_skip_adapter_distill', 'data_root': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/Data', 'student_checkpoint_path': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/192606_2404_v5_clean_no_block_residual/checkpoints/best.pth', 'teacher_checkpoint_path': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/162928_2404_v5_c44_b5_mixed_psnr/checkpoints/best.pth', 'config': {'model_id': 'hinet_lite_npu_v6c_stable_skip_adapter_distill', 'run_slug': 'v6c_stable_skip_adapter_distill', 'seed': 1337, 'eval_size': 256, 'crop_size': 256, 'batch_size': 8, 'epochs': 120, 'lr': 0.0001, 'min_lr': 5e-06, 'weight_decay': 5e-05, 'warmup_epochs': 5, 'grad_clip_norm': 0.5, 'mse_weight': 0.9, 'l1_weight': 0.1, 'distill_weight': 0.05, 'adapter_freeze_epochs': 3, 'early_stop_patience': 20, 'num_workers': 2, 'use_amp': False, '

## 2. Data

The required Lab 3 contract for this HiNet Colab path is L3 paired training data under `LR_HR/L3_HR_train*` and `L3_LR/L3_*`, validation data under `L3_HR_valid` and `L3_LR_valid`, and `3x256x256` tensors. Calibration images later in the notebook are copied from training LR images only.


In [4]:
L3_TRAIN_SPLITS = (
    ('LR_HR/L3_HR_train1', 'L3_LR/L3_L3_train1', 'L3_HR_train1'),
    ('LR_HR/L3_HR_train2', 'L3_LR/L3_L3_train2', 'L3_HR_train2'),
    ('LR_HR/L3_HR_train3', 'L3_LR/L3_LR_train3', 'L3_HR_train3'),
    ('LR_HR/L3_HR_train4', 'L3_LR/L3_LR_train4', 'L3_HR_train4'),
    ('LR_HR/L3_HR_train', 'L3_LR/L3_LR_train', 'L3_HR_train'),
)
L3_VAL_DIRS = ('L3_HR_valid', 'L3_LR_valid')


def scan_image_directory(directory: Path) -> dict[str, Path]:
    if not directory.exists():
        raise FileNotFoundError(f'Missing directory: {directory}')
    return {path.stem: path for path in directory.iterdir() if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES}


def collect_split_pairs(hr_dir: Path, lr_dir: Path, split_name: str) -> list[tuple[Path, Path, str]]:
    hr_map = scan_image_directory(hr_dir)
    lr_map = scan_image_directory(lr_dir)
    pairs: list[tuple[Path, Path, str]] = []
    for stem in sorted(set(hr_map) & set(lr_map)):
        name = stem if split_name == 'val' else f'{split_name}/{stem}'
        pairs.append((lr_map[stem], hr_map[stem], name))
    return pairs


def collect_all_pairs(data_root: Path) -> tuple[list[tuple[Path, Path, str]], list[tuple[Path, Path, str]]]:
    train_pairs: list[tuple[Path, Path, str]] = []
    for hr_rel, lr_rel, split_name in L3_TRAIN_SPLITS:
        train_pairs.extend(collect_split_pairs(data_root / hr_rel, data_root / lr_rel, split_name))
    val_pairs = collect_split_pairs(data_root / L3_VAL_DIRS[0], data_root / L3_VAL_DIRS[1], 'val')
    return train_pairs, val_pairs


class PairedSRDataset(Dataset):
    def __init__(self, pairs: list[tuple[Path, Path, str]], *, train: bool, crop_size: int, eval_size: int, seed: int):
        self.pairs = pairs
        self.train = train
        self.crop_size = crop_size
        self.eval_size = eval_size
        self.seed = seed
        self.epoch = 0

    def set_epoch(self, epoch: int) -> None:
        self.epoch = epoch

    @staticmethod
    def to_tensor(image: Image.Image) -> torch.Tensor:
        array = np.asarray(image, dtype=np.float32) / 255.0
        return torch.from_numpy(array).permute(2, 0, 1).contiguous()

    def augment_pair(self, lr_image: Image.Image, hr_image: Image.Image, index: int) -> tuple[Image.Image, Image.Image]:
        rng = random.Random((self.seed * 1_000_003) + (self.epoch * 9_973) + index)
        width, height = lr_image.size
        if width < self.crop_size or height < self.crop_size:
            lr_image = lr_image.resize((self.eval_size, self.eval_size), BICUBIC_RESAMPLE)
            hr_image = hr_image.resize((self.eval_size, self.eval_size), BICUBIC_RESAMPLE)
            width, height = lr_image.size
        left = rng.randint(0, width - self.crop_size)
        top = rng.randint(0, height - self.crop_size)
        box = (left, top, left + self.crop_size, top + self.crop_size)
        lr_image = lr_image.crop(box)
        hr_image = hr_image.crop(box)
        if rng.random() < 0.5:
            lr_image = lr_image.transpose(FLIP_LEFT_RIGHT)
            hr_image = hr_image.transpose(FLIP_LEFT_RIGHT)
        if rng.random() < 0.5:
            lr_image = lr_image.transpose(FLIP_TOP_BOTTOM)
            hr_image = hr_image.transpose(FLIP_TOP_BOTTOM)
        rot = rng.choice([0, 1, 2, 3])
        if rot:
            rotate_ops = [ROTATE_90, ROTATE_180, ROTATE_270]
            lr_image = lr_image.transpose(rotate_ops[rot - 1])
            hr_image = hr_image.transpose(rotate_ops[rot - 1])
        return lr_image, hr_image

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, index: int) -> dict[str, Any]:
        lr_path, hr_path, name = self.pairs[index]
        lr_image = Image.open(lr_path).convert('RGB')
        hr_image = Image.open(hr_path).convert('RGB')
        if self.train:
            lr_image, hr_image = self.augment_pair(lr_image, hr_image, index)
        elif lr_image.size != (self.eval_size, self.eval_size):
            lr_image = lr_image.resize((self.eval_size, self.eval_size), BICUBIC_RESAMPLE)
            hr_image = hr_image.resize((self.eval_size, self.eval_size), BICUBIC_RESAMPLE)
        return {'lr': self.to_tensor(lr_image), 'hr': self.to_tensor(hr_image), 'name': name}


train_pairs, val_pairs = collect_all_pairs(DRIVE_DATA_ROOT)
raw_pair_counts = {'train_pairs': len(train_pairs), 'val_pairs': len(val_pairs)}
if len(train_pairs) != EXPECTED_TRAIN_PAIRS or len(val_pairs) != EXPECTED_VAL_PAIRS:
    print({'pair_count_warning': {**raw_pair_counts, 'expected_train_pairs': EXPECTED_TRAIN_PAIRS, 'expected_val_pairs': EXPECTED_VAL_PAIRS}})
if cfg.train_pair_limit is not None:
    train_pairs = train_pairs[: cfg.train_pair_limit]
if cfg.val_pair_limit is not None:
    val_pairs = val_pairs[: cfg.val_pair_limit]
if not train_pairs or not val_pairs:
    raise RuntimeError(f'Expected non-empty L3 train/val pairs, found train={len(train_pairs)} val={len(val_pairs)}')

train_ds = PairedSRDataset(train_pairs, train=True, crop_size=cfg.crop_size, eval_size=cfg.eval_size, seed=cfg.seed)
val_ds = PairedSRDataset(val_pairs, train=False, crop_size=cfg.eval_size, eval_size=cfg.eval_size, seed=cfg.seed)
loader_kwargs = {'num_workers': cfg.num_workers, 'pin_memory': device.type == 'cuda'}
if cfg.num_workers > 0:
    loader_kwargs['persistent_workers'] = True
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, drop_last=False, **loader_kwargs)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, drop_last=False, **loader_kwargs)
train_sample = train_ds[0]
val_sample = val_ds[0]
shape_contract = {
    'train_lr_shape': tuple(train_sample['lr'].shape),
    'train_hr_shape': tuple(train_sample['hr'].shape),
    'val_lr_shape': tuple(val_sample['lr'].shape),
    'val_hr_shape': tuple(val_sample['hr'].shape),
}
expected_shape = (3, 256, 256)
if any(shape != expected_shape for shape in shape_contract.values()):
    raise RuntimeError(f'Lab 3 tensor shape contract failed: {shape_contract}')
print({'raw_pair_counts': raw_pair_counts, 'effective_train_pairs': len(train_pairs), 'effective_val_pairs': len(val_pairs), **shape_contract})


{'raw_pair_counts': {'train_pairs': 2217, 'val_pairs': 110}, 'effective_train_pairs': 2217, 'effective_val_pairs': 110, 'train_lr_shape': (3, 256, 256), 'train_hr_shape': (3, 256, 256), 'val_lr_shape': (3, 256, 256), 'val_hr_shape': (3, 256, 256)}


## 3. Model and Training

The v6c student keeps the v6b forward graph unchanged: no internal block residual adds, identity-initialized `1x1` skip adapters on the two U-Net skip paths, and a global residual output. The training recipe is changed to reduce numerical risk: lower learning rate, AMP disabled, weaker residual distillation, a short adapter/tail warmup stage, and strict finite-value checkpoint gates.


In [5]:
@dataclass
class HiNetLiteV6CSkipAdapterConfig:
    channels: int = 44
    encoder_blocks: tuple[int, int] = (3, 3)
    bottleneck_blocks: int = 5
    decoder_blocks: tuple[int, int] = (3, 3)
    kernel_pattern: tuple[int, ...] = (3, 5, 3, 5, 3, 5)
    residual_scale: float = 0.2
    use_block_residual: bool = False
    use_unet_skips: bool = True
    global_residual: bool = True


@dataclass
class HiNetLiteV5TeacherConfig:
    channels: int = 44
    encoder_blocks: tuple[int, int] = (3, 3)
    bottleneck_blocks: int = 5
    decoder_blocks: tuple[int, int] = (3, 3)
    kernel_pattern: tuple[int, ...] = (3, 5, 3, 5, 3, 5)
    residual_scale: float = 0.2
    use_block_residual: bool = True
    use_unet_skips: bool = True
    global_residual: bool = True


def init_tail_small(conv: nn.Conv2d, scale: float = 1e-3) -> None:
    nn.init.kaiming_normal_(conv.weight, a=0.1, mode='fan_in', nonlinearity='leaky_relu')
    conv.weight.data.mul_(scale)
    if conv.bias is not None:
        nn.init.zeros_(conv.bias)


@torch.no_grad()
def init_identity_1x1(conv: nn.Conv2d) -> None:
    if conv.kernel_size != (1, 1):
        raise ValueError('Identity initialization requires a 1x1 Conv2d.')
    nn.init.zeros_(conv.weight)
    if conv.bias is not None:
        nn.init.zeros_(conv.bias)
    diag = min(conv.out_channels, conv.in_channels)
    for idx in range(diag):
        conv.weight[idx, idx, 0, 0] = 1.0


class MixedKernelCleanBlock(nn.Module):
    def __init__(self, channels: int, kernel_size: int, residual_scale: float, use_block_residual: bool):
        super().__init__()
        padding = kernel_size // 2
        self.residual_scale = float(residual_scale)
        self.use_block_residual = bool(use_block_residual)
        self.scale_folded = False
        self.conv1 = nn.Conv2d(channels, channels, kernel_size, 1, padding)
        self.act = nn.LeakyReLU(0.1, inplace=True)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size, 1, padding)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = self.conv2(self.act(self.conv1(x)))
        if not self.use_block_residual:
            return residual
        if self.scale_folded:
            return x + residual
        return x + self.residual_scale * residual

    @torch.no_grad()
    def fold_residual_scale_for_export(self) -> None:
        if not self.use_block_residual or self.scale_folded:
            return
        self.conv2.weight.mul_(self.residual_scale)
        if self.conv2.bias is not None:
            self.conv2.bias.mul_(self.residual_scale)
        self.scale_folded = True
        self.residual_scale = 1.0


class DownsampleBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = nn.Sequential(nn.Conv2d(in_ch, out_ch, 3, stride=2, padding=1), nn.LeakyReLU(0.1, inplace=True))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class UpsampleBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = nn.Sequential(nn.ConvTranspose2d(in_ch, out_ch, 4, stride=2, padding=1), nn.LeakyReLU(0.1, inplace=True))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class HiNetLiteV6CSkipAdapterSR(nn.Module):
    def __init__(self, model_cfg: HiNetLiteV6CSkipAdapterConfig):
        super().__init__()
        c = model_cfg.channels
        pattern = model_cfg.kernel_pattern
        self.use_unet_skips = model_cfg.use_unet_skips
        self.global_residual = model_cfg.global_residual
        self.stem = nn.Sequential(nn.Conv2d(3, c, 3, 1, 1), nn.LeakyReLU(0.1, inplace=True))
        self.enc1 = nn.Sequential(*[MixedKernelCleanBlock(c, pattern[idx % len(pattern)], model_cfg.residual_scale, model_cfg.use_block_residual) for idx in range(model_cfg.encoder_blocks[0])])
        self.down1 = DownsampleBlock(c, c * 2)
        self.enc2 = nn.Sequential(*[MixedKernelCleanBlock(c * 2, pattern[(model_cfg.encoder_blocks[0] + idx) % len(pattern)], model_cfg.residual_scale, model_cfg.use_block_residual) for idx in range(model_cfg.encoder_blocks[1])])
        self.down2 = DownsampleBlock(c * 2, c * 4)
        self.bottleneck = nn.Sequential(*[MixedKernelCleanBlock(c * 4, pattern[(sum(model_cfg.encoder_blocks) + idx) % len(pattern)], model_cfg.residual_scale, model_cfg.use_block_residual) for idx in range(model_cfg.bottleneck_blocks)])
        self.up1 = UpsampleBlock(c * 4, c * 2)
        self.skip2_proj = nn.Conv2d(c * 2, c * 2, 1, 1, 0)
        init_identity_1x1(self.skip2_proj)
        self.dec1 = nn.Sequential(*[MixedKernelCleanBlock(c * 2, pattern[(sum(model_cfg.encoder_blocks) + model_cfg.bottleneck_blocks + idx) % len(pattern)], model_cfg.residual_scale, model_cfg.use_block_residual) for idx in range(model_cfg.decoder_blocks[0])])
        self.up2 = UpsampleBlock(c * 2, c)
        self.skip1_proj = nn.Conv2d(c, c, 1, 1, 0)
        init_identity_1x1(self.skip1_proj)
        self.dec2 = nn.Sequential(*[MixedKernelCleanBlock(c, pattern[(sum(model_cfg.encoder_blocks) + model_cfg.bottleneck_blocks + model_cfg.decoder_blocks[0] + idx) % len(pattern)], model_cfg.residual_scale, model_cfg.use_block_residual) for idx in range(model_cfg.decoder_blocks[1])])
        self.tail = nn.Conv2d(c, 3, 3, 1, 1)
        init_tail_small(self.tail)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        stem = self.stem(x)
        enc1 = self.enc1(stem)
        enc2 = self.enc2(self.down1(enc1))
        bottleneck = self.bottleneck(self.down2(enc2))
        up1 = self.up1(bottleneck)
        dec1_input = up1 + self.skip2_proj(enc2) if self.use_unet_skips else up1
        dec1 = self.dec1(dec1_input)
        up2 = self.up2(dec1)
        dec2_input = up2 + self.skip1_proj(enc1) if self.use_unet_skips else up2
        dec2 = self.dec2(dec2_input)
        delta = self.tail(dec2)
        return x + delta if self.global_residual else delta

    @torch.no_grad()
    def fold_residual_scales_for_export(self) -> None:
        for module in self.modules():
            if isinstance(module, MixedKernelCleanBlock):
                module.fold_residual_scale_for_export()


class HiNetLiteV5TeacherSR(nn.Module):
    def __init__(self, model_cfg: HiNetLiteV5TeacherConfig):
        super().__init__()
        c = model_cfg.channels
        pattern = model_cfg.kernel_pattern
        self.use_unet_skips = model_cfg.use_unet_skips
        self.global_residual = model_cfg.global_residual
        self.stem = nn.Sequential(nn.Conv2d(3, c, 3, 1, 1), nn.LeakyReLU(0.1, inplace=True))
        self.enc1 = nn.Sequential(*[MixedKernelCleanBlock(c, pattern[idx % len(pattern)], model_cfg.residual_scale, model_cfg.use_block_residual) for idx in range(model_cfg.encoder_blocks[0])])
        self.down1 = DownsampleBlock(c, c * 2)
        self.enc2 = nn.Sequential(*[MixedKernelCleanBlock(c * 2, pattern[(model_cfg.encoder_blocks[0] + idx) % len(pattern)], model_cfg.residual_scale, model_cfg.use_block_residual) for idx in range(model_cfg.encoder_blocks[1])])
        self.down2 = DownsampleBlock(c * 2, c * 4)
        self.bottleneck = nn.Sequential(*[MixedKernelCleanBlock(c * 4, pattern[(sum(model_cfg.encoder_blocks) + idx) % len(pattern)], model_cfg.residual_scale, model_cfg.use_block_residual) for idx in range(model_cfg.bottleneck_blocks)])
        self.up1 = UpsampleBlock(c * 4, c * 2)
        self.dec1 = nn.Sequential(*[MixedKernelCleanBlock(c * 2, pattern[(sum(model_cfg.encoder_blocks) + model_cfg.bottleneck_blocks + idx) % len(pattern)], model_cfg.residual_scale, model_cfg.use_block_residual) for idx in range(model_cfg.decoder_blocks[0])])
        self.up2 = UpsampleBlock(c * 2, c)
        self.dec2 = nn.Sequential(*[MixedKernelCleanBlock(c, pattern[(sum(model_cfg.encoder_blocks) + model_cfg.bottleneck_blocks + model_cfg.decoder_blocks[0] + idx) % len(pattern)], model_cfg.residual_scale, model_cfg.use_block_residual) for idx in range(model_cfg.decoder_blocks[1])])
        self.tail = nn.Conv2d(c, 3, 3, 1, 1)
        init_tail_small(self.tail)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        stem = self.stem(x)
        enc1 = self.enc1(stem)
        enc2 = self.enc2(self.down1(enc1))
        bottleneck = self.bottleneck(self.down2(enc2))
        up1 = self.up1(bottleneck)
        dec1_input = up1 + enc2 if self.use_unet_skips else up1
        dec1 = self.dec1(dec1_input)
        up2 = self.up2(dec1)
        dec2_input = up2 + enc1 if self.use_unet_skips else up2
        dec2 = self.dec2(dec2_input)
        delta = self.tail(dec2)
        return x + delta if self.global_residual else delta


_ALLOWED_SKIP_MISSING = {'skip1_proj.weight', 'skip1_proj.bias', 'skip2_proj.weight', 'skip2_proj.bias'}


def count_parameters(module: nn.Module) -> dict[str, int]:
    total = sum(param.numel() for param in module.parameters())
    trainable = sum(param.numel() for param in module.parameters() if param.requires_grad)
    return {'total': int(total), 'trainable': int(trainable)}


def select_checkpoint_state(checkpoint: dict[str, Any], preferred: str) -> tuple[dict[str, torch.Tensor], str]:
    if preferred == 'ema' and checkpoint.get('ema_model') is not None:
        return checkpoint['ema_model'], 'ema'
    if checkpoint.get('model') is not None:
        return checkpoint['model'], 'model'
    if checkpoint.get('state_dict') is not None:
        return checkpoint['state_dict'], 'state_dict'
    raise KeyError('Checkpoint does not contain model, ema_model, or state_dict weights.')


def load_student_initialization(student_model: nn.Module, checkpoint_path: Path | None) -> dict[str, Any]:
    if checkpoint_path is None or not checkpoint_path.exists():
        raise FileNotFoundError(f'v6 student checkpoint is required: {checkpoint_path}')
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    source_state, source_name = select_checkpoint_state(checkpoint, cfg.selected_weight_source)
    incompatible = student_model.load_state_dict(source_state, strict=False)
    missing_keys = list(incompatible.missing_keys)
    unexpected_keys = list(incompatible.unexpected_keys)
    unsupported_missing = sorted(set(missing_keys) - _ALLOWED_SKIP_MISSING)
    if unexpected_keys or unsupported_missing:
        raise RuntimeError({'unexpected_keys': unexpected_keys, 'unsupported_missing_keys': unsupported_missing})
    return {
        'path': str(checkpoint_path),
        'source': source_name,
        'strict': False,
        'missing_keys': missing_keys,
        'unexpected_keys': unexpected_keys,
        'skip_adapters_identity_initialized': sorted(_ALLOWED_SKIP_MISSING),
        'checkpoint_epoch': checkpoint.get('epoch'),
        'checkpoint_best_val_psnr': checkpoint.get('best_val_psnr'),
    }


def load_teacher_model(checkpoint_path: Path | None, runtime_device: torch.device) -> tuple[nn.Module | None, dict[str, Any]]:
    if checkpoint_path is None or not checkpoint_path.exists():
        info = {'enabled': False, 'reason': 'teacher_checkpoint_missing', 'path': str(checkpoint_path), 'weight': 0.0}
        if cfg.allow_missing_teacher:
            print(info)
            return None, info
        raise FileNotFoundError(f'v5 teacher checkpoint is required when allow_missing_teacher=False: {checkpoint_path}')
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    defaults = asdict(HiNetLiteV5TeacherConfig())
    checkpoint_cfg = checkpoint.get('model_config', {}) or {}
    defaults.update({key: value for key, value in checkpoint_cfg.items() if key in defaults})
    teacher_cfg = HiNetLiteV5TeacherConfig(**defaults)
    teacher = HiNetLiteV5TeacherSR(teacher_cfg).to(runtime_device).eval()
    source_state, source_name = select_checkpoint_state(checkpoint, cfg.selected_weight_source)
    teacher.load_state_dict(source_state, strict=True)
    for param in teacher.parameters():
        param.requires_grad_(False)
    info = {
        'enabled': bool(cfg.distill_weight > 0.0),
        'path': str(checkpoint_path),
        'source': source_name,
        'weight': float(cfg.distill_weight),
        'model_config': asdict(teacher_cfg),
        'checkpoint_epoch': checkpoint.get('epoch'),
        'checkpoint_best_val_psnr': checkpoint.get('best_val_psnr'),
    }
    return teacher if info['enabled'] else None, info


model_cfg = HiNetLiteV6CSkipAdapterConfig()
model = HiNetLiteV6CSkipAdapterSR(model_cfg).to(device)
student_init_info = load_student_initialization(model, student_checkpoint_path)
teacher_model, teacher_info = load_teacher_model(teacher_checkpoint_path, device)
model_num_parameters = count_parameters(model)
print({'model_name': model.__class__.__name__, 'model_config': asdict(model_cfg), 'expected_input_output_shape': (1, 3, 256, 256), 'parameter_count': model_num_parameters, 'student_initialization': student_init_info, 'teacher_distillation': teacher_info})


{'model_name': 'HiNetLiteV6CSkipAdapterSR', 'model_config': {'channels': 44, 'encoder_blocks': (3, 3), 'bottleneck_blocks': 5, 'decoder_blocks': (3, 3), 'kernel_pattern': (3, 5, 3, 5, 3, 5), 'residual_scale': 0.2, 'use_block_residual': False, 'use_unet_skips': True, 'global_residual': True}, 'expected_input_output_shape': (1, 3, 256, 256), 'parameter_count': {'total': 7430855, 'trainable': 7430855}, 'student_initialization': {'path': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/192606_2404_v5_clean_no_block_residual/checkpoints/best.pth', 'source': 'ema', 'strict': False, 'missing_keys': ['skip2_proj.weight', 'skip2_proj.bias', 'skip1_proj.weight', 'skip1_proj.bias'], 'unexpected_keys': [], 'skip_adapters_identity_initialized': ['skip1_proj.bias', 'skip1_proj.weight', 'skip2_proj.bias', 'skip2_proj.weight'], 'checkpoint_epoch': 81, 'checkpoint_best_val_psnr': 23.930102400346236}, 'teacher_distillation': {'enabled': True, 'path': '/content/drive/MyDrive/Data 25

In [ ]:
def psnr_from_mse(mse: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    return 10.0 * torch.log10(1.0 / torch.clamp(mse, min=eps))


@torch.no_grad()
def batch_psnr(pred: torch.Tensor, target: torch.Tensor) -> float:
    mse = F.mse_loss(pred, target, reduction='none').mean(dim=(1, 2, 3))
    return float(psnr_from_mse(mse).mean().item())


def move_image_batch(batch: torch.Tensor) -> torch.Tensor:
    return batch.to(device, non_blocking=(device.type == 'cuda'))


def scalar_is_finite(value: Any) -> bool:
    try:
        return math.isfinite(float(value))
    except (TypeError, ValueError):
        return False


def metrics_are_finite(metrics: dict[str, Any], keys: list[str]) -> bool:
    return all(scalar_is_finite(metrics.get(key)) for key in keys)


@torch.no_grad()
def tensor_is_finite(tensor: torch.Tensor) -> bool:
    return bool(torch.isfinite(tensor.detach()).all().item())


@torch.no_grad()
def module_state_is_finite(module: nn.Module) -> bool:
    return all(tensor_is_finite(param) for param in module.state_dict().values())


@torch.no_grad()
def ema_state_is_finite(ema: 'EMA | None') -> bool:
    if ema is None:
        return True
    return all(tensor_is_finite(value) for value in ema.shadow.values())


def train_stage_for_epoch(epoch: int) -> str:
    return 'adapter_tail_warmup' if epoch <= cfg.adapter_freeze_epochs else 'full_finetune'


def apply_train_stage(epoch: int) -> dict[str, Any]:
    stage = train_stage_for_epoch(epoch)
    trainable_prefixes = ('skip1_proj.', 'skip2_proj.', 'tail.')
    for name, param in model.named_parameters():
        param.requires_grad_(stage == 'full_finetune' or name.startswith(trainable_prefixes))
    return {'train_stage': stage, **count_parameters(model)}


def restoration_loss(pred: torch.Tensor, lr: torch.Tensor, hr: torch.Tensor, teacher_pred: torch.Tensor | None) -> dict[str, torch.Tensor]:
    pred_res = pred - lr
    target_res = hr - lr
    loss_mse = F.mse_loss(pred_res, target_res)
    loss_l1 = F.l1_loss(pred_res, target_res)
    loss_distill = pred.new_tensor(0.0)
    if teacher_pred is not None and cfg.distill_weight > 0.0:
        teacher_res = teacher_pred.detach() - lr
        loss_distill = F.l1_loss(pred_res, teacher_res)
    loss = cfg.mse_weight * loss_mse + cfg.l1_weight * loss_l1 + cfg.distill_weight * loss_distill
    return {'loss': loss, 'loss_mse': loss_mse.detach(), 'loss_l1': loss_l1.detach(), 'loss_distill': loss_distill.detach()}


def lr_at_epoch(epoch_idx: int) -> float:
    step = epoch_idx + 1
    if cfg.warmup_epochs > 0 and step <= cfg.warmup_epochs:
        return cfg.lr * step / max(1, cfg.warmup_epochs)
    progress = max(0.0, (step - cfg.warmup_epochs) / max(1, cfg.epochs - cfg.warmup_epochs))
    cosine = 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))
    return cfg.min_lr + cosine * (cfg.lr - cfg.min_lr)


def autocast_context(enabled: bool):
    if not enabled:
        return nullcontext()
    if hasattr(torch, 'amp') and hasattr(torch.amp, 'autocast'):
        return torch.amp.autocast(device_type=device.type, enabled=enabled)
    return torch.cuda.amp.autocast(enabled=enabled)


def make_grad_scaler(enabled: bool):
    if hasattr(torch, 'amp') and hasattr(torch.amp, 'GradScaler'):
        return torch.amp.GradScaler('cuda', enabled=enabled)
    return torch.cuda.amp.GradScaler(enabled=enabled)


class EMA:
    def __init__(self, source_model: nn.Module, decay: float):
        self.decay = decay
        self.shadow = {key: value.detach().clone() for key, value in source_model.state_dict().items()}

    @torch.no_grad()
    def update(self, source_model: nn.Module) -> None:
        for key, value in source_model.state_dict().items():
            self.shadow[key].mul_(self.decay).add_(value.detach(), alpha=1.0 - self.decay)

    @torch.no_grad()
    def copy_to(self, target_model: nn.Module) -> None:
        target_model.load_state_dict(self.shadow, strict=True)


@torch.inference_mode()
def evaluate_psnr(eval_model: nn.Module, loader: DataLoader) -> dict[str, float]:
    eval_model.eval()
    pred_meter = 0.0
    input_meter = 0.0
    count = 0
    for batch in loader:
        lr = move_image_batch(batch['lr'])
        hr = move_image_batch(batch['hr'])
        pred = eval_model(lr).clamp(0, 1)
        pred_meter += batch_psnr(pred, hr)
        input_meter += batch_psnr(lr, hr)
        count += 1
    return {'val_psnr': pred_meter / max(count, 1), 'input_psnr': input_meter / max(count, 1), 'delta_psnr': (pred_meter - input_meter) / max(count, 1)}


def train_one_epoch(epoch: int, optimizer: torch.optim.Optimizer, scaler: Any, ema: EMA | None) -> dict[str, Any]:
    model.train()
    train_ds.set_epoch(epoch)
    amp_enabled = bool(cfg.use_amp and device.type == 'cuda')
    stage_info = apply_train_stage(epoch)
    loss_meter = 0.0
    distill_meter = 0.0
    psnr_meter = 0.0
    count = 0
    nonfinite_step_count = 0
    skipped_step_count = 0
    optimizer_step_count = 0
    post_step_nonfinite_count = 0
    grad_norm_meter = 0.0
    guard_reasons: Counter[str] = Counter()

    for batch in train_loader:
        lr = move_image_batch(batch['lr'])
        hr = move_image_batch(batch['hr'])
        optimizer.zero_grad(set_to_none=True)
        teacher_pred = None
        with autocast_context(amp_enabled):
            if teacher_model is not None:
                with torch.no_grad():
                    teacher_pred = teacher_model(lr).clamp(0, 1)
                if not tensor_is_finite(teacher_pred):
                    nonfinite_step_count += 1
                    skipped_step_count += 1
                    guard_reasons['teacher_pred_nonfinite'] += 1
                    continue
            pred = model(lr)
            loss_info = restoration_loss(pred, lr, hr, teacher_pred)
            loss = loss_info['loss']
        if not tensor_is_finite(loss):
            nonfinite_step_count += 1
            skipped_step_count += 1
            guard_reasons['loss_nonfinite'] += 1
            continue
        if not module_state_is_finite(model):
            nonfinite_step_count += 1
            skipped_step_count += 1
            guard_reasons['pre_step_model_state_nonfinite'] += 1
            continue
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)
        grad_norm_value = float(grad_norm.detach().cpu().item() if isinstance(grad_norm, torch.Tensor) else grad_norm)
        if not math.isfinite(grad_norm_value):
            optimizer.zero_grad(set_to_none=True)
            nonfinite_step_count += 1
            skipped_step_count += 1
            guard_reasons['grad_norm_nonfinite'] += 1
            continue
        scaler.step(optimizer)
        scaler.update()
        optimizer_step_count += 1
        grad_norm_meter += grad_norm_value
        if not module_state_is_finite(model):
            nonfinite_step_count += 1
            post_step_nonfinite_count += 1
            guard_reasons['post_step_model_state_nonfinite'] += 1
            break
        if ema is not None:
            ema.update(model)
            if not ema_state_is_finite(ema):
                nonfinite_step_count += 1
                post_step_nonfinite_count += 1
                guard_reasons['ema_state_nonfinite'] += 1
                break
        with torch.no_grad():
            loss_meter += float(loss.item())
            distill_meter += float(loss_info['loss_distill'].item())
            psnr_meter += batch_psnr(pred.clamp(0, 1), hr)
            count += 1

    return {
        'train_loss': loss_meter / count if count else float('nan'),
        'train_distill_loss': distill_meter / count if count else float('nan'),
        'train_psnr': psnr_meter / count if count else float('nan'),
        'train_stage': stage_info['train_stage'],
        'trainable_parameters': int(stage_info['trainable']),
        'nonfinite_step_count': int(nonfinite_step_count),
        'skipped_step_count': int(skipped_step_count),
        'optimizer_step_count': int(optimizer_step_count),
        'post_step_nonfinite_count': int(post_step_nonfinite_count),
        'mean_grad_norm': grad_norm_meter / optimizer_step_count if optimizer_step_count else float('nan'),
        'guard_reasons': dict(guard_reasons),
    }


def build_eval_model(ema: EMA | None) -> nn.Module:
    eval_model = HiNetLiteV6CSkipAdapterSR(model_cfg).to(device).eval()
    if cfg.selected_weight_source == 'ema' and ema is not None:
        ema.copy_to(eval_model)
    else:
        eval_model.load_state_dict(model.state_dict(), strict=True)
    return eval_model


def save_checkpoint(path: Path, epoch: int, optimizer: torch.optim.Optimizer, scaler: Any, ema: EMA | None, metrics: dict[str, Any], best_val_psnr: float, best_epoch: int, stop_reason: str) -> None:
    payload = {
        'epoch': epoch,
        'model': model.state_dict(),
        'ema_model': ema.shadow if ema is not None else None,
        'optimizer': optimizer.state_dict(),
        'scaler': scaler.state_dict() if scaler.is_enabled() else None,
        'config': asdict(cfg),
        'model_config': asdict(model_cfg),
        'selected_weight_source': cfg.selected_weight_source,
        'student_initialization': student_init_info,
        'teacher_distillation': teacher_info,
        'train_schedule': {'adapter_freeze_epochs': cfg.adapter_freeze_epochs, 'full_finetune_starts_epoch': cfg.adapter_freeze_epochs + 1},
        'best_val_psnr': float(best_val_psnr),
        'best_epoch': int(best_epoch),
        'stop_reason': stop_reason,
        'last_metrics': metrics,
    }
    torch.save(payload, path)


optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scaler = make_grad_scaler(enabled=(cfg.use_amp and device.type == 'cuda'))
ema = EMA(model, cfg.ema_decay) if cfg.use_ema else None
best_ckpt_path = checkpoints_dir / 'best.pth'
last_ckpt_path = checkpoints_dir / 'last.pth'
metrics_csv_path = run_root / 'metrics.csv'
history_fields = [
    'epoch', 'lr', 'train_stage', 'trainable_parameters', 'train_loss', 'train_distill_loss', 'train_psnr',
    'val_psnr', 'input_psnr', 'delta_psnr', 'distill_enabled', 'nonfinite_step_count', 'skipped_step_count',
    'optimizer_step_count', 'post_step_nonfinite_count', 'mean_grad_norm', 'checkpoint_eligible', 'saved_best'
]
best_val_psnr = float('-inf')
best_epoch = 0
epochs_without_improvement = 0
stop_reason = 'completed_max_epochs'
finite_guard_totals = {'nonfinite_step_count': 0, 'skipped_step_count': 0, 'post_step_nonfinite_count': 0, 'guard_reasons': Counter()}
train_schedule_info = {'adapter_freeze_epochs': cfg.adapter_freeze_epochs, 'full_finetune_starts_epoch': cfg.adapter_freeze_epochs + 1}

with metrics_csv_path.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=history_fields)
    writer.writeheader()

for epoch in range(1, cfg.epochs + 1):
    epoch_lr = lr_at_epoch(epoch - 1)
    for group in optimizer.param_groups:
        group['lr'] = epoch_lr
    train_metrics = train_one_epoch(epoch, optimizer, scaler, ema)
    eval_model = build_eval_model(ema)
    val_metrics = evaluate_psnr(eval_model, val_loader)
    model_finite = module_state_is_finite(model)
    ema_finite = ema_state_is_finite(ema)
    train_finite = metrics_are_finite(train_metrics, ['train_loss', 'train_distill_loss', 'train_psnr'])
    val_finite = metrics_are_finite(val_metrics, ['val_psnr', 'input_psnr', 'delta_psnr'])
    checkpoint_eligible = bool(train_finite and val_finite and model_finite and ema_finite and train_metrics['nonfinite_step_count'] == 0)
    is_best = bool(checkpoint_eligible and val_metrics['val_psnr'] > best_val_psnr)
    saved_best = False
    row = {
        'epoch': epoch,
        'lr': epoch_lr,
        **{key: value for key, value in train_metrics.items() if key != 'guard_reasons'},
        **val_metrics,
        'distill_enabled': bool(teacher_model is not None),
        'checkpoint_eligible': checkpoint_eligible,
        'saved_best': is_best,
    }
    finite_guard_totals['nonfinite_step_count'] += int(train_metrics['nonfinite_step_count'])
    finite_guard_totals['skipped_step_count'] += int(train_metrics['skipped_step_count'])
    finite_guard_totals['post_step_nonfinite_count'] += int(train_metrics['post_step_nonfinite_count'])
    finite_guard_totals['guard_reasons'].update(train_metrics.get('guard_reasons', {}))
    if is_best:
        best_val_psnr = float(val_metrics['val_psnr'])
        best_epoch = epoch
        epochs_without_improvement = 0
        save_checkpoint(best_ckpt_path, epoch, optimizer, scaler, ema, row, best_val_psnr, best_epoch, stop_reason)
        saved_best = True
        row['saved_best'] = True
    else:
        epochs_without_improvement += 1
    if checkpoint_eligible:
        save_checkpoint(last_ckpt_path, epoch, optimizer, scaler, ema, row, best_val_psnr, best_epoch, stop_reason)
    with metrics_csv_path.open('a', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=history_fields)
        writer.writerow({key: row[key] for key in history_fields})
    print({
        'epoch': epoch,
        'lr': epoch_lr,
        'train_stage': train_metrics['train_stage'],
        'train_loss': train_metrics['train_loss'],
        'train_distill_loss': train_metrics['train_distill_loss'],
        'train_psnr': train_metrics['train_psnr'],
        'val_psnr': val_metrics['val_psnr'],
        'input_psnr': val_metrics['input_psnr'],
        'delta_psnr': val_metrics['delta_psnr'],
        'best_val_psnr': best_val_psnr,
        'distill_enabled': bool(teacher_model is not None),
        'nonfinite_step_count': train_metrics['nonfinite_step_count'],
        'checkpoint_eligible': checkpoint_eligible,
        'saved_best': saved_best,
        'epochs_without_improvement': epochs_without_improvement,
    })
    if not train_finite:
        stop_reason = f'nonfinite_train_metrics_epoch_{epoch}'
    elif not val_finite:
        stop_reason = f'nonfinite_validation_metrics_epoch_{epoch}'
    elif not model_finite:
        stop_reason = f'nonfinite_model_state_epoch_{epoch}'
    elif not ema_finite:
        stop_reason = f'nonfinite_ema_state_epoch_{epoch}'
    elif train_metrics['post_step_nonfinite_count'] > 0:
        stop_reason = f'post_step_nonfinite_state_epoch_{epoch}'
    elif epoch > cfg.warmup_epochs and train_metrics['nonfinite_step_count'] > 0:
        stop_reason = f'nonfinite_step_after_warmup_epoch_{epoch}'
    elif epochs_without_improvement >= cfg.early_stop_patience:
        stop_reason = f'early_stop_no_val_improvement_{cfg.early_stop_patience}'
    if stop_reason != 'completed_max_epochs':
        if checkpoint_eligible:
            save_checkpoint(last_ckpt_path, epoch, optimizer, scaler, ema, row, best_val_psnr, best_epoch, stop_reason)
        print({'stop_reason': stop_reason, 'epoch': epoch, 'best_epoch': best_epoch, 'best_val_psnr': best_val_psnr, 'finite_guard_totals': {**finite_guard_totals, 'guard_reasons': dict(finite_guard_totals['guard_reasons'])}})
        break

if not best_ckpt_path.exists():
    raise RuntimeError('No finite eligible best checkpoint was saved; v6c training did not produce an exportable candidate.')
finite_guard_summary = {**finite_guard_totals, 'guard_reasons': dict(finite_guard_totals['guard_reasons'])}
print({'best_checkpoint': str(best_ckpt_path), 'best_epoch': best_epoch, 'best_val_psnr': best_val_psnr, 'stop_reason': stop_reason, 'finite_guard_summary': finite_guard_summary})


{'epoch': 1, 'lr': 2e-05, 'train_stage': 'adapter_tail_warmup', 'train_loss': 0.00848929941480001, 'train_distill_loss': 0.011794056096817735, 'train_psnr': 25.125747865910153, 'val_psnr': 23.93058669350364, 'input_psnr': 23.133060299266468, 'delta_psnr': 0.7975263942371715, 'best_val_psnr': 23.93058669350364, 'distill_enabled': True, 'nonfinite_step_count': 0, 'checkpoint_eligible': True, 'saved_best': True, 'epochs_without_improvement': 0}
{'epoch': 2, 'lr': 4e-05, 'train_stage': 'adapter_tail_warmup', 'train_loss': 0.008491260847143776, 'train_distill_loss': 0.011728272743174712, 'train_psnr': 25.117207492855812, 'val_psnr': 23.931046919389203, 'input_psnr': 23.133060299266468, 'delta_psnr': 0.7979866201227361, 'best_val_psnr': 23.931046919389203, 'distill_enabled': True, 'nonfinite_step_count': 0, 'checkpoint_eligible': True, 'saved_best': True, 'epochs_without_improvement': 0}
{'epoch': 3, 'lr': 6.000000000000001e-05, 'train_stage': 'adapter_tail_warmup', 'train_loss': 0.008484787

## 4. Validation

Reload the selected best weights, compute validation PSNR, and write preview strips as `LR | prediction | HR`.


In [ ]:
def load_best_eval_model(checkpoint_path: Path, runtime_device: torch.device) -> nn.Module:
    checkpoint = torch.load(checkpoint_path, map_location=runtime_device)
    defaults = asdict(model_cfg)
    checkpoint_cfg = checkpoint.get('model_config', defaults) or defaults
    defaults.update({key: value for key, value in checkpoint_cfg.items() if key in defaults})
    loaded_model_cfg = HiNetLiteV6CSkipAdapterConfig(**defaults)
    eval_model = HiNetLiteV6CSkipAdapterSR(loaded_model_cfg).to(runtime_device).eval()
    source_state, _ = select_checkpoint_state(checkpoint, checkpoint.get('selected_weight_source', cfg.selected_weight_source))
    eval_model.load_state_dict(source_state, strict=True)
    return eval_model


def tensor_to_uint8_image(tensor: torch.Tensor) -> Image.Image:
    array = tensor.detach().clamp(0, 1).mul(255.0).round().byte().permute(1, 2, 0).cpu().numpy()
    return Image.fromarray(array, mode='RGB')


@torch.inference_mode()
def write_val_preview(eval_model: nn.Module, loader: DataLoader, out_dir: Path, max_items: int = 8) -> dict[str, Any]:
    out_dir.mkdir(parents=True, exist_ok=True)
    eval_model.eval()
    items: list[dict[str, str]] = []
    for batch in loader:
        lr = move_image_batch(batch['lr'])
        hr = move_image_batch(batch['hr'])
        pred = eval_model(lr).clamp(0, 1)
        batch_name = batch['name'][0] if isinstance(batch['name'], (list, tuple)) else str(batch['name'])
        safe_name = str(batch_name).replace('/', '_').replace('\\', '_')
        lr_img = tensor_to_uint8_image(lr[0])
        pred_img = tensor_to_uint8_image(pred[0])
        hr_img = tensor_to_uint8_image(hr[0])
        canvas = Image.new('RGB', (cfg.eval_size * 3, cfg.eval_size))
        canvas.paste(lr_img, (0, 0))
        canvas.paste(pred_img, (cfg.eval_size, 0))
        canvas.paste(hr_img, (cfg.eval_size * 2, 0))
        out_path = out_dir / f'{len(items):02d}_{safe_name}.png'
        canvas.save(out_path)
        items.append({'name': str(batch_name), 'preview_path': str(out_path), 'layout': 'LR | prediction | HR'})
        if len(items) >= max_items:
            break
    return {'preview_dir': str(out_dir), 'count': len(items), 'items': items}


best_eval_model = load_best_eval_model(best_ckpt_path, device)
validation_summary = evaluate_psnr(best_eval_model, val_loader)
validation_recheck = {'finite': metrics_are_finite(validation_summary, ['val_psnr', 'input_psnr', 'delta_psnr']), 'matches_recorded_best': bool(abs(validation_summary['val_psnr'] - best_val_psnr) < 1e-5), 'abs_diff_vs_recorded_best': float(abs(validation_summary['val_psnr'] - best_val_psnr))}
if not validation_recheck['finite'] or not validation_recheck['matches_recorded_best']:
    raise RuntimeError({'validation_recheck': validation_recheck, 'best_val_psnr': best_val_psnr, 'validation_summary': validation_summary})
preview_info = write_val_preview(best_eval_model, val_loader, exports_dir / 'val_preview')
print({'validation_summary': validation_summary, 'val_preview': preview_info, 'best_checkpoint': str(best_ckpt_path), 'best_epoch': best_epoch, 'validation_recheck': validation_recheck, 'beats_v6b_reference': validation_summary['val_psnr'] > V6B_REFERENCE_BEST_VAL_PSNR, 'beats_v6_baseline': validation_summary['val_psnr'] > V6_REFERENCE_BEST_VAL_PSNR, 'beats_hinet_colab_v5': validation_summary['val_psnr'] > V5_REFERENCE_BEST_VAL_PSNR})


## 5. Export and Submission Artifacts

Export fixed-shape `1x3x256x256` ONNX, audit NPU-sensitive ops, run ONNX Runtime parity when available, write calibration PNGs from training LR images, and create an MXQ conversion handoff script.


In [ ]:
FORBIDDEN_OPS = {'Mul', 'Concat', 'Resize', 'MatMul', 'Softmax', 'Sigmoid', 'Tanh'}
ALLOWED_EXPORT_OPS = {'Conv', 'ConvTranspose', 'LeakyRelu', 'Add'}
SUSPICIOUS_OPS = FORBIDDEN_OPS | {'Add', 'InstanceNormalization', 'GroupNormalization', 'LayerNormalization', 'ReduceMean', 'Div', 'Sub', 'Pow', 'Sqrt', 'Reshape', 'Transpose', 'Slice', 'Gather', 'Unsqueeze', 'Squeeze'}


def audit_onnx_graph(onnx_path: Path) -> dict[str, Any]:
    if onnx is None:
        return {'checked': False, 'reason': 'onnx unavailable'}
    loaded = onnx.load(str(onnx_path))
    onnx.checker.check_model(loaded)
    ops = [node.op_type for node in loaded.graph.node]
    counts = dict(sorted(Counter(ops).items()))
    suspicious = {op: counts[op] for op in sorted(SUSPICIOUS_OPS) if op in counts}
    forbidden = {op: counts[op] for op in sorted(FORBIDDEN_OPS) if op in counts}
    return {'checked': True, 'onnx_checker': 'passed', 'total_node_count': len(ops), 'unique_op_types': sorted(counts), 'op_counts': counts, 'suspicious_ops_found': suspicious, 'forbidden_ops_found': forbidden, 'mul_found': 'Mul' in counts, 'add_count': int(counts.get('Add', 0)), 'add_count_ok': int(counts.get('Add', 0)) <= 3, 'unexpected_ops_found': {op: counts[op] for op in sorted(counts) if op not in ALLOWED_EXPORT_OPS}, 'allowed_export_ops': sorted(ALLOWED_EXPORT_OPS)}


export_model = load_best_eval_model(best_ckpt_path, device)
export_model.eval()
fold_input = torch.randn(1, 3, 256, 256, device=device)
with torch.inference_mode():
    before_fold = export_model(fold_input).detach()
export_model.fold_residual_scales_for_export()
with torch.inference_mode():
    after_fold = export_model(fold_input).detach()
folding_summary = {'fold_max_abs_diff': float((before_fold - after_fold).abs().max().item()), 'fold_mean_abs_diff': float((before_fold - after_fold).abs().mean().item())}
print(folding_summary)

onnx_path = exports_dir / f'{cfg.model_id}.onnx'
dummy = torch.randn(1, 3, 256, 256, device=device)
torch.onnx.export(export_model, dummy, str(onnx_path), input_names=['input'], output_names=['output'], opset_version=13, do_constant_folding=True, dynamic_axes=None, dynamo=False)
op_audit = audit_onnx_graph(onnx_path)
print({'onnx_path': str(onnx_path), **op_audit})

parity_result = {'attempted': False, 'onnxruntime_available': ort is not None, 'passed': False, 'max_abs_diff': None, 'mean_abs_diff': None}
if ort is not None:
    cpu_model = load_best_eval_model(best_ckpt_path, torch.device('cpu')).cpu().eval()
    cpu_model.fold_residual_scales_for_export()
    parity_input = torch.randn(1, 3, 256, 256)
    with torch.no_grad():
        torch_out = cpu_model(parity_input).detach().cpu().numpy()
    session = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
    ort_out = session.run(['output'], {'input': parity_input.numpy()})[0]
    diff = np.abs(torch_out - ort_out)
    parity_result.update({'attempted': True, 'passed': bool(float(diff.max()) < 1e-3), 'max_abs_diff': float(diff.max()), 'mean_abs_diff': float(diff.mean())})
print(parity_result)

onnx_acceptance = {'mul_absent': not bool(op_audit.get('mul_found')), 'forbidden_ops_absent': not bool(op_audit.get('forbidden_ops_found')), 'add_count_lte_3': bool(op_audit.get('add_count_ok')), 'unexpected_ops_absent': not bool(op_audit.get('unexpected_ops_found')), 'parity_max_abs_diff_lt_1e-3': bool(parity_result.get('passed'))}
print({'onnx_acceptance': onnx_acceptance})


In [ ]:
def write_calibration_bundle(train_pairs: list[tuple[Path, Path, str]], out_dir: Path, max_items: int) -> dict[str, Any]:
    out_dir.mkdir(parents=True, exist_ok=True)
    selected = train_pairs[: max_items]
    manifest = {'source': 'L3 training LR images', 'derived_from_training': True, 'count': len(selected), 'items': []}
    for idx, (lr_path, _, name) in enumerate(selected):
        dst = out_dir / f'cal_{idx:04d}.png'
        image = Image.open(lr_path).convert('RGB')
        if image.size != (cfg.eval_size, cfg.eval_size):
            image = image.resize((cfg.eval_size, cfg.eval_size), BICUBIC_RESAMPLE)
        image.save(dst)
        manifest['items'].append({'index': idx, 'name': name, 'lr_source': str(lr_path), 'calibration_path': str(dst)})
    manifest_path = out_dir / 'manifest.json'
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    return {'calibration_dir': str(out_dir), 'manifest_path': str(manifest_path), 'count': len(selected), 'derived_from_training': True}


conversion_script_path = exports_dir / 'convert_v6c_mxq.py'
conversion_script = r'''#!/usr/bin/env python3
from __future__ import annotations
import argparse, json, shlex, shutil, subprocess
from pathlib import Path

def find_tool():
    for name in ['mxq_compile', 'qubee', 'qb']:
        path = shutil.which(name)
        if path:
            return path
    return None

parser = argparse.ArgumentParser(description='ONNX to MXQ helper for HiNetLite v6c stable skip-adapter student.')
parser.add_argument('--onnx', type=Path, required=True)
parser.add_argument('--calibration-dir', type=Path, required=True)
parser.add_argument('--output', type=Path, required=True)
parser.add_argument('--command-template', default='')
parser.add_argument('--dry-run', action='store_true')
args = parser.parse_args()
args.onnx = args.onnx.resolve()
args.calibration_dir = args.calibration_dir.resolve()
args.output = args.output.resolve()
if not args.onnx.exists():
    raise FileNotFoundError(args.onnx)
if not args.calibration_dir.exists():
    raise FileNotFoundError(args.calibration_dir)
tool = find_tool()
payload = {'onnx': str(args.onnx), 'calibration_dir': str(args.calibration_dir), 'output': str(args.output), 'mxq_tool_detected': tool, 'status': 'handoff_only'}
if args.command_template:
    command = args.command_template.format(onnx=shlex.quote(str(args.onnx)), calibration=shlex.quote(str(args.calibration_dir)), output=shlex.quote(str(args.output)))
elif tool:
    command = ' '.join([shlex.quote(tool), shlex.quote(str(args.onnx)), shlex.quote(str(args.calibration_dir)), shlex.quote(str(args.output))])
else:
    print(json.dumps(payload, indent=2))
    raise SystemExit(0)
payload['command'] = command
payload['status'] = 'dry_run' if args.dry_run else 'running'
if args.dry_run:
    print(json.dumps(payload, indent=2))
    raise SystemExit(0)
completed = subprocess.run(command, shell=True, capture_output=True, text=True)
payload.update({'returncode': completed.returncode, 'stdout': completed.stdout, 'stderr': completed.stderr, 'status': 'completed' if completed.returncode == 0 else 'failed', 'output_exists': args.output.exists()})
print(json.dumps(payload, indent=2))
raise SystemExit(completed.returncode)
'''
conversion_script_path.write_text(conversion_script, encoding='utf-8')
calibration_info = write_calibration_bundle(train_pairs, calibration_dir, cfg.calibration_count)
mxq_target_path = exports_dir / f'{cfg.model_id}.mxq'
mxq_command = [sys.executable, str(conversion_script_path), '--onnx', str(onnx_path), '--calibration-dir', str(calibration_dir), '--output', str(mxq_target_path)]
if cfg.mxq_command_template:
    mxq_command.extend(['--command-template', cfg.mxq_command_template])
if not cfg.run_mxq_compile:
    mxq_command.append('--dry-run')
mxq_handoff = {'conversion_script_path': str(conversion_script_path), 'onnx_input_path': str(onnx_path), 'calibration_manifest_path': calibration_info['manifest_path'], 'mxq_target_path': str(mxq_target_path), 'command': ' '.join(shlex.quote(part) for part in mxq_command), 'requested_compile': cfg.run_mxq_compile}
print({'calibration': calibration_info, 'mxq_handoff': mxq_handoff})


## 6. Final Summary

This cell writes `summary.json` plus a Markdown report comparing v6c against v6b, the v6 baseline, hinet_colab_v5, the v4 reference, and the 25 dB course target. It also records finite-guard counters and whether the run is promotable under the v6c criteria.


In [ ]:
comparisons = {
    'course_target_val_psnr': 25.0,
    'v6b_reference_best_val_psnr': V6B_REFERENCE_BEST_VAL_PSNR,
    'v6_baseline_best_val_psnr': V6_REFERENCE_BEST_VAL_PSNR,
    'hinet_colab_v5_best_val_psnr': V5_REFERENCE_BEST_VAL_PSNR,
    'v4_reference_best_val_psnr': V4_REFERENCE_BEST_VAL_PSNR,
    'delta_vs_course_target': float(best_val_psnr - 25.0),
    'delta_vs_v6b_reference': float(best_val_psnr - V6B_REFERENCE_BEST_VAL_PSNR),
    'delta_vs_v6_baseline': float(best_val_psnr - V6_REFERENCE_BEST_VAL_PSNR),
    'delta_vs_hinet_colab_v5': float(best_val_psnr - V5_REFERENCE_BEST_VAL_PSNR),
    'delta_vs_v4_reference': float(best_val_psnr - V4_REFERENCE_BEST_VAL_PSNR),
    'beats_course_target_25db': bool(best_val_psnr > 25.0),
    'beats_v6b_reference': bool(best_val_psnr > V6B_REFERENCE_BEST_VAL_PSNR),
    'beats_v6_baseline': bool(best_val_psnr > V6_REFERENCE_BEST_VAL_PSNR),
    'beats_hinet_colab_v5': bool(best_val_psnr > V5_REFERENCE_BEST_VAL_PSNR),
    'beats_v4_reference': bool(best_val_psnr > V4_REFERENCE_BEST_VAL_PSNR),
}
acceptance = {
    'no_nonfinite_steps': int(finite_guard_summary.get('nonfinite_step_count', 0)) == 0,
    'psnr_beats_v6b_reference': comparisons['beats_v6b_reference'],
    'psnr_beats_hinet_colab_v5_preferred_target': comparisons['beats_hinet_colab_v5'],
    'psnr_beats_25db_course_target': comparisons['beats_course_target_25db'],
    'onnx_mul_absent': onnx_acceptance['mul_absent'],
    'onnx_forbidden_ops_absent': onnx_acceptance['forbidden_ops_absent'],
    'onnx_unexpected_ops_absent': onnx_acceptance['unexpected_ops_absent'],
    'onnx_add_count_lte_3': onnx_acceptance['add_count_lte_3'],
    'onnx_runtime_parity_max_abs_diff_lt_1e-3': onnx_acceptance['parity_max_abs_diff_lt_1e-3'],
    'calibration_derived_from_training': bool(calibration_info.get('derived_from_training')),
    'mxq_handoff_present': bool(mxq_handoff.get('conversion_script_path') and mxq_handoff.get('mxq_target_path')),
    'artifact_chain_present': all(bool(path) for path in [str(best_ckpt_path), str(onnx_path), calibration_info['manifest_path'], str(conversion_script_path), str(mxq_target_path)]),
}
acceptance['promotable_v6c_candidate'] = bool(
    acceptance['no_nonfinite_steps']
    and acceptance['psnr_beats_v6b_reference']
    and acceptance['onnx_mul_absent']
    and acceptance['onnx_forbidden_ops_absent']
    and acceptance['onnx_unexpected_ops_absent']
    and acceptance['onnx_add_count_lte_3']
    and acceptance['onnx_runtime_parity_max_abs_diff_lt_1e-3']
    and acceptance['calibration_derived_from_training']
    and acceptance['mxq_handoff_present']
)
summary = {
    'model_id': cfg.model_id,
    'variant': cfg.run_slug,
    'run_root': str(run_root),
    'config': asdict(cfg),
    'model_config': asdict(model_cfg),
    'parameter_count': model_num_parameters,
    'student_initialization': student_init_info,
    'teacher_distillation': teacher_info,
    'train_schedule': train_schedule_info,
    'finite_guard_summary': finite_guard_summary,
    'validation': validation_summary,
    'validation_recheck': validation_recheck,
    'best_epoch': int(best_epoch),
    'best_val_psnr': float(best_val_psnr),
    'stop_reason': stop_reason,
    'references': {
        'course_target': {'best_val_psnr': 25.0},
        'v6b_reference': {'model': 'hinet_lite_npu_v6b_skip_adapter_256patch_distill', 'best_val_psnr': V6B_REFERENCE_BEST_VAL_PSNR},
        'v6_baseline': {'model': 'hinet_lite_npu_v6_v5_clean_no_block_residual', 'best_val_psnr': V6_REFERENCE_BEST_VAL_PSNR},
        'hinet_colab_v5': {'model': 'hinet_colab_v5_c44_b5_mixed_psnr', 'best_val_psnr': V5_REFERENCE_BEST_VAL_PSNR},
        'v4_reference': {'model': 'hinet_lite_npu_v4_reference', 'best_val_psnr': V4_REFERENCE_BEST_VAL_PSNR},
    },
    'comparisons': comparisons,
    'acceptance': acceptance,
    'best_checkpoint_path': str(best_ckpt_path),
    'last_checkpoint_path': str(last_ckpt_path),
    'onnx_path': str(onnx_path),
    'folding_summary': folding_summary,
    'onnx_parity': parity_result,
    'op_audit': op_audit,
    'onnx_acceptance': onnx_acceptance,
    'calibration': calibration_info,
    'val_preview': preview_info,
    'mxq_handoff': mxq_handoff,
    'artifact_paths': {
        'pth': str(best_ckpt_path),
        'onnx': str(onnx_path),
        'calibration_manifest': calibration_info['manifest_path'],
        'conversion_script': str(conversion_script_path),
        'mxq_target': str(mxq_target_path),
    },
}
summary_path = run_root / 'summary.json'
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
report_path = run_root / 'v6c_experiment_summary.md'
report_text = f'''# HiNetLite NPU v6c - stable skip-adapter distill

| Field | Value |
| --- | --- |
| Variant | `{cfg.run_slug}` |
| Parameter count | `{model_num_parameters['total']}` |
| Best validation PSNR | `{best_val_psnr:.4f}` |
| Input baseline PSNR | `{validation_summary['input_psnr']:.4f}` |
| Delta vs 25 dB target | `{comparisons['delta_vs_course_target']:.4f}` |
| Delta vs v6b reference | `{comparisons['delta_vs_v6b_reference']:.4f}` |
| Delta vs v6 baseline | `{comparisons['delta_vs_v6_baseline']:.4f}` |
| Delta vs hinet_colab_v5 | `{comparisons['delta_vs_hinet_colab_v5']:.4f}` |
| Delta vs v4 reference | `{comparisons['delta_vs_v4_reference']:.4f}` |
| Train schedule | `{train_schedule_info}` |
| Finite guard summary | `{finite_guard_summary}` |
| Student checkpoint | `{student_init_info['path']}` |
| Teacher distillation | `{teacher_info}` |
| Best checkpoint | `{best_ckpt_path}` |
| ONNX path | `{onnx_path}` |
| ONNX node count | `{op_audit.get('total_node_count')}` |
| ONNX Add count | `{op_audit.get('add_count')}` |
| Unexpected ONNX ops | `{op_audit.get('unexpected_ops_found')}` |
| Forbidden ONNX ops | `{op_audit.get('forbidden_ops_found')}` |
| ONNX parity max diff | `{parity_result.get('max_abs_diff')}` |
| Calibration manifest | `{calibration_info['manifest_path']}` |
| MXQ target | `{mxq_target_path}` |
| Promotable v6c candidate | `{acceptance['promotable_v6c_candidate']}` |

## Acceptance

```json
{json.dumps(acceptance, indent=2)}
```
'''
report_path.write_text(report_text, encoding='utf-8')
print({'summary_path': str(summary_path), 'report_path': str(report_path), 'artifacts': summary['artifact_paths'], 'acceptance': acceptance})
